<a href="https://colab.research.google.com/github/rrstats/CalendarUp/blob/main/CalendarDates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# PDF to Google Calendar Link Generator
# Install required packages
!pip install PyPDF2 pdfplumber transformers torch dateparser pandas -q

import pdfplumber
import PyPDF2
import re
from datetime import datetime, timedelta
from urllib.parse import quote
import dateparser
from transformers import pipeline
import pandas as pd
from google.colab import files
import io

class PDFToCalendar:
    def __init__(self):
        print("Loading NLP model for event extraction...")
        # Using a simple NER model to help extract entities
        try:
            self.nlp = pipeline("ner", model="dslim/bert-base-NER")
        except:
            print("NLP model not loaded, will use pattern matching only")
            self.nlp = None
        print("Ready!")

    def upload_pdf(self):
        """Upload PDF file"""
        print("📤 Please upload your PDF file:")
        uploaded = files.upload()

        if not uploaded:
            raise ValueError("No file uploaded")

        filename = list(uploaded.keys())[0]
        return filename, uploaded[filename]

    def extract_text_from_pdf(self, pdf_bytes):
        """Extract text from PDF"""
        print("\n📄 Extracting text from PDF...")

        text_lines = []

        # Try with pdfplumber first (better for structured PDFs)
        try:
            pdf_file = io.BytesIO(pdf_bytes)
            with pdfplumber.open(pdf_file) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        text_lines.extend(text.split('\n'))
        except Exception as e:
            print(f"pdfplumber failed: {e}")

            # Fallback to PyPDF2
            try:
                pdf_file = io.BytesIO(pdf_bytes)
                pdf_reader = PyPDF2.PdfReader(pdf_file)
                for page in pdf_reader.pages:
                    text = page.extract_text()
                    if text:
                        text_lines.extend(text.split('\n'))
            except Exception as e2:
                print(f"PyPDF2 also failed: {e2}")
                raise

        # Clean up lines
        text_lines = [line.strip() for line in text_lines if line.strip()]
        print(f"✓ Extracted {len(text_lines)} lines")

        return text_lines

    def parse_date_time(self, text):
        """Parse date and time from text"""
        # Try dateparser first
        parsed = dateparser.parse(text, settings={
            'PREFER_DATES_FROM': 'future',
            'RETURN_AS_TIMEZONE_AWARE': False
        })

        if parsed:
            return parsed

        # Manual patterns for common formats
        patterns = [
            r'(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',  # 12/25/2024 or 12-25-24
            r'(\d{4}[/-]\d{1,2}[/-]\d{1,2})',    # 2024-12-25
            r'((?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{1,2},?\s+\d{4})',  # January 15, 2024
            r'(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4})',    # 15 January 2024
        ]

        for pattern in patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                try:
                    return dateparser.parse(match.group(1))
                except:
                    continue

        return None

    def parse_time(self, text):
        """Extract time from text"""
        # Common time patterns
        time_patterns = [
            r'(\d{1,2}:\d{2}\s*(?:AM|PM|am|pm))',  # 2:30 PM
            r'(\d{1,2}:\d{2})',                     # 14:30
            r'(\d{1,2}\s*(?:AM|PM|am|pm))',        # 2 PM
        ]

        for pattern in time_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                return match.group(1)

        return None

    def extract_events(self, text_lines):
        """Extract event information from text lines"""
        print("\n🔍 Parsing events...")

        events = []
        current_event = {}

        for i, line in enumerate(text_lines):
            # Skip empty lines
            if not line:
                continue

            # Try to parse date
            date = self.parse_date_time(line)

            # Try to parse time
            time = self.parse_time(line)

            # Check if this line looks like an event title (heuristic)
            is_title = (
                len(line.split()) <= 10 and  # Not too long
                not any(word in line.lower() for word in ['the', 'and', 'for', 'with']) and
                line[0].isupper()  # Starts with capital
            )

            # Build event
            if date or time:
                if current_event and current_event.get('title'):
                    events.append(current_event)
                    current_event = {}

                if date:
                    current_event['date'] = date
                if time:
                    current_event['time'] = time
                current_event['raw_line'] = line

            elif is_title and not current_event.get('title'):
                current_event['title'] = line
                current_event['raw_line'] = line

            elif current_event and not current_event.get('description'):
                # Add as description
                current_event['description'] = line

            # Also try to parse as structured line (e.g., "Event Name | Date | Time")
            if '|' in line or '\t' in line:
                parts = re.split(r'[|\t]', line)
                if len(parts) >= 2:
                    event = {'raw_line': line}

                    for part in parts:
                        part = part.strip()
                        if not part:
                            continue

                        date = self.parse_date_time(part)
                        time = self.parse_time(part)

                        if date:
                            event['date'] = date
                        elif time:
                            event['time'] = time
                        elif not event.get('title'):
                            event['title'] = part
                        elif not event.get('description'):
                            event['description'] = part

                    if event.get('title'):
                        events.append(event)
                        current_event = {}

        # Add last event
        if current_event and current_event.get('title'):
            events.append(current_event)

        # Filter and clean events
        cleaned_events = []
        for event in events:
            if event.get('title'):
                # Set default date to today if missing
                if not event.get('date'):
                    event['date'] = datetime.now()

                # Set default time if missing
                if not event.get('time'):
                    event['time'] = '10:00 AM'

                cleaned_events.append(event)

        print(f"✓ Found {len(cleaned_events)} potential events")
        return cleaned_events

    def create_google_calendar_link(self, title, start_datetime, end_datetime=None,
                                   description="", location=""):
        """Create a Google Calendar link"""
        if end_datetime is None:
            end_datetime = start_datetime + timedelta(hours=1)

        # Format: YYYYMMDDTHHmmSS
        start_str = start_datetime.strftime("%Y%m%dT%H%M%S")
        end_str = end_datetime.strftime("%Y%m%dT%H%M%S")

        # Build URL
        base_url = "https://calendar.google.com/calendar/render?action=TEMPLATE"

        params = {
            'text': title,
            'dates': f"{start_str}/{end_str}",
            'details': description,
            'location': location
        }

        url = base_url
        for key, value in params.items():
            if value:
                url += f"&{key}={quote(str(value))}"

        return url

    def process_events(self, events):
        """Process events and create calendar links"""
        print("\n📅 Creating Google Calendar links...\n")

        calendar_data = []

        for i, event in enumerate(events, 1):
            title = event.get('title', f'Event {i}')
            date = event.get('date', datetime.now())
            time_str = event.get('time', '10:00 AM')
            description = event.get('description', '')

            # Combine date and time
            try:
                # Parse time and combine with date
                time_parsed = dateparser.parse(time_str)
                if time_parsed:
                    start_datetime = date.replace(
                        hour=time_parsed.hour,
                        minute=time_parsed.minute
                    )
                else:
                    start_datetime = date.replace(hour=10, minute=0)
            except:
                start_datetime = date.replace(hour=10, minute=0)

            # Create calendar link
            calendar_link = self.create_google_calendar_link(
                title=title,
                start_datetime=start_datetime,
                description=description
            )

            calendar_data.append({
                'Event': title,
                'Date': start_datetime.strftime("%Y-%m-%d"),
                'Time': start_datetime.strftime("%I:%M %p"),
                'Description': description,
                'Calendar Link': calendar_link
            })

            print(f"{i}. {title}")
            print(f"   📅 {start_datetime.strftime('%B %d, %Y at %I:%M %p')}")
            print(f"   🔗 {calendar_link}")
            print()

        return calendar_data

    def save_to_csv(self, calendar_data):
        """Save calendar links to CSV"""
        df = pd.DataFrame(calendar_data)
        csv_filename = 'calendar_events.csv'
        df.to_csv(csv_filename, index=False)
        print(f"\n💾 Saved to {csv_filename}")
        files.download(csv_filename)
        return df

    def run(self):
        """Main execution flow"""
        print("="*60)
        print("📄 PDF to Google Calendar Link Generator")
        print("="*60)

        # Upload PDF
        filename, pdf_bytes = self.upload_pdf()

        # Extract text
        text_lines = self.extract_text_from_pdf(pdf_bytes)

        # Show extracted text
        print("\n📝 Extracted Text:")
        print("-" * 60)
        for i, line in enumerate(text_lines[:20], 1):  # Show first 20 lines
            print(f"{i}. {line}")
        if len(text_lines) > 20:
            print(f"... and {len(text_lines) - 20} more lines")
        print("-" * 60)

        # Extract events
        events = self.extract_events(text_lines)

        if not events:
            print("\n⚠️  No events found. The PDF might not contain recognizable event data.")
            print("Please ensure your PDF has event information in a structured format.")
            return

        # Create calendar links
        calendar_data = self.process_events(events)

        # Save to CSV
        df = self.save_to_csv(calendar_data)

        print("\n✅ Done! Click the calendar links above to add events to Google Calendar.")

        return df

# Run the converter
converter = PDFToCalendar()
result_df = converter.run()

# Display as table
if result_df is not None:
    print("\n" + "="*60)
    print("📊 Summary Table")
    print("="*60)
    display(result_df)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 67.7 MB/s eta 0:00:00
Loading NLP model for event extraction...


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

...

Saving JHU AGAI January.pdf to JHU AGAI January.pdf

📄 Extracting text from PDF...
✓ Extracted 46 lines

📝 Extracted Text:
------------------------------------------------------------
1. Certificate Program in Applied Generative AI
2. Learning Schedule - January 2026 Cohort
3. Program Content Live Session³
4. Course # Course Topic JHU Master Classes Assessment Deadline⁴
5. Week Release Weekend
6. Available on
7. Pre-work-01: Introduction to the World of AI
8. enrollment
9. Available on
10. 0 Pre Work¹ 0 Pre-work 02: Overview of Generative AI
11. enrollment
12. Available on
13. Pre-work 03: Python Foundation
14. enrollment
15. Preparatory Session 10-Jan
16. - - - Program Orientation² 17-Jan
17. 0 MLS 0 Introduction To Python 24-Jan --
18. 1 Generative AI Landscape 22-Jan 31-Jan -- 1-Feb
19. Learning Python with
20. 1
... and 26 more lines
------------------------------------------------------------

🔍 Parsing events...
✓ Found 3 potential events

📅 Creating Google Calendar links...

1. 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Done! Click the calendar links above to add events to Google Calendar.

📊 Summary Table


,Event,Date,Time,Description,Calendar Link
0,Certificate Program in Applied Generative AI,2026-01-01,10:00 AM,Learning Schedule - January 2026 Cohort,https://calendar.google.com/calendar/render?ac...
1,Gen AI,2027-01-01,10:00 AM,Generative AI 2 Python Programming with Genera...,https://calendar.google.com/calendar/render?ac...
2,Generative AI Workflows 14 Advanced RAG 16-Apr...,2026-03-01,10:00 AM,15 Fine Tuning and Customization of Generative...,https://calendar.google.com/calendar/render?ac...
